# Customer Segmentation & Retention Analysis using Online Retail II Dataset

## Project Overview

Customer segmentation is the process of grouping customers based on similar purchasing behavior. Retention analysis focuses on understanding customer loyalty and identifying customers who are at risk of becoming inactive.

In this project, transaction-level data from an online retail company will be analyzed to:

- Understand customer purchasing behavior.
- Segment customers using RFM Analysis and K-Means Clustering.
- Analyze customer retention patterns.
- Generate actionable business recommendations.
- Build an interactive dashboard for business users.

This project follows an end-to-end data science workflow, beginning with raw transaction data and ending with business insights and deployment.

# Business Problem

E-commerce companies collect millions of transaction records every year. However, not all customers contribute equally to business growth.

Some customers purchase frequently and generate high revenue, while others purchase only once and never return.

Without understanding customer behavior, businesses face several challenges:

- Difficulty identifying valuable customers.
- Inefficient marketing campaigns.
- Poor customer retention.
- Low return on marketing investment.

The objective of this project is to analyze historical transaction data to identify meaningful customer segments, evaluate customer retention, and provide recommendations that improve business performance.

# Project Objectives

The primary objectives of this project are:

1. Explore and understand customer transaction data.
2. Clean and preprocess raw retail data.
3. Perform exploratory data analysis (EDA).
4. Engineer customer-level features using RFM Analysis.
5. Segment customers using K-Means Clustering.
6. Analyze customer retention behavior.
7. Generate business recommendations for each customer segment.
8. Develop an interactive dashboard for visualization and decision-making.

# Phase 2: Data Loading

## Objective

The Online Retail II dataset contains transaction records from a UK-based online retailer between December 2009 and December 2011.

For efficient processing, the dataset has been converted into a consolidated CSV file containing the complete transaction history.

The dataset is loaded into a pandas DataFrame, which will be used throughout this project for data cleaning, exploratory analysis, feature engineering, customer segmentation, and retention analysis.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/online_retail_II.csv")

In [3]:
plt.style.use("default")

# Phase 3: Data Understanding

## Objective

Before cleaning or analyzing the data, it is essential to understand its structure, quality, and business context.

The Online Retail II dataset contains transactional records from a UK-based online retailer. Each row represents a product purchased within a customer invoice.

Understanding the available features allows us to identify potential data quality issues and determine which variables are relevant for customer segmentation and retention analysis.

In [4]:
print(df.shape)
df.info()

(1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [5]:
df.describe(include="all")


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371,1067371,1062989,1.067371e+06,1067371,1.067371e+06,824364.000000,1067371
unique,53628,5305,5698,NaN,47635,NaN,NaN,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,2010-12-06 16:57:00,NaN,NaN,United Kingdom
freq,1350,5829,5918,NaN,1350,NaN,NaN,981330
mean,NaN,NaN,NaN,9.938898e+00,NaN,4.649388e+00,15324.638504,NaN
std,NaN,NaN,NaN,1.727058e+02,NaN,1.235531e+02,1697.464450,NaN
min,NaN,NaN,NaN,-8.099500e+04,NaN,-5.359436e+04,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000e+00,NaN,1.250000e+00,13975.000000,NaN
50%,NaN,NaN,NaN,3.000000e+00,NaN,2.100000e+00,15255.000000,NaN
75%,NaN,NaN,NaN,1.000000e+01,NaN,4.150000e+00,16797.000000,NaN


## Observations

The initial inspection of the dataset reveals several important characteristics:

- The dataset contains **1,067,371 transaction records** across **8 features**.
- Transaction data spans multiple countries, with the United Kingdom appearing most frequently.
- The **Customer ID** column contains a significant number of missing values, which will require careful handling since customer-level analysis depends on this field.
- A small number of missing values are also present in the **Description** column.
- The **InvoiceDate** column is currently stored as a string and will be converted into a datetime format during preprocessing.
- Negative values are observed in both **Quantity** and **Price**, suggesting product returns, cancellations, or erroneous records that must be investigated before analysis.

In [6]:
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
df.tail()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France


# Phase 4: Data Cleaning

## Objective

Real-world business datasets often contain missing values, duplicate records, cancelled transactions, invalid entries, and inconsistent data types.

The objective of this phase is to improve the quality of the dataset before performing exploratory analysis and machine learning.

Each cleaning decision will be based on business understanding rather than simply removing data.

## Data Cleaning Strategy

Before modifying the dataset, each data quality issue is investigated individually.

Instead of applying generic cleaning methods, every decision is supported by business reasoning.

The following aspects will be examined:

- Missing values
- Duplicate records
- Data types
- Negative quantities
- Negative prices
- Customer identifiers

Only after understanding the cause of each issue will appropriate cleaning actions be applied.


In [8]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [9]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum()/len(df))*100
})

missing

,Missing Values,Percentage
Invoice,0,0.000000
StockCode,0,0.000000
Description,4382,0.410541
Quantity,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Customer ID,243007,22.766873
Country,0,0.000000


In [10]:
df.duplicated().sum()

np.int64(34335)

## Convert Invoice Date

In [11]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [13]:
df[df["Quantity"] < 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [14]:
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [15]:
df[df["Customer ID"].isnull()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom


## Data Cleaning Decisions

After investigating the dataset, the following cleaning strategy was adopted.

| Data Quality Issue | Decision | Business Justification |
|--------------------|----------|------------------------|
| Missing Product Description | Retained | Product descriptions are not required for customer-level analysis. |
| Missing Customer ID | Removed | Customer segmentation and retention analysis require a unique customer identifier. |
| Duplicate Records | Removed | Duplicate transactions would incorrectly inflate purchase frequency and revenue. |
| Negative Quantity | Removed | Negative quantities represent returned or cancelled orders rather than completed purchases. |
| Negative Price | Removed | Negative prices correspond to accounting adjustments and are not valid customer purchases. |
| InvoiceDate Data Type | Converted to datetime | Required for calculating purchase recency and retention metrics. |

In [16]:
print("Original Shape:", df.shape)

Original Shape: (1067371, 8)


In [17]:
df = df.dropna(subset=["Customer ID"])

In [18]:
df = df.drop_duplicates()

In [19]:
df = df[df["Quantity"] > 0]

In [20]:
df = df[df["Price"] > 0]

In [21]:
df.reset_index(drop=True, inplace=True)

In [22]:
print("Cleaned Shape:", df.shape)

df.head()

Cleaned Shape: (779425, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [23]:
df.shape


(779425, 8)

In [24]:
df.isnull().sum()


Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [25]:
df.duplicated().sum()


np.int64(0)

In [26]:
(df["Quantity"] <= 0).sum()


np.int64(0)

In [27]:
(df["Price"] <= 0).sum()

np.int64(0)

## Save Cleaned Dataset

The cleaned transaction-level dataset is saved to disk so that it can be reused by the
downstream notebooks (`02_EDA.ipynb`, `03_RFM_Analysis.ipynb`, ...) without repeating the
cleaning steps above. This keeps each notebook focused on a single phase of the pipeline
and guarantees every phase starts from the exact same cleaned data.

In [28]:
from pathlib import Path

Path("../data").mkdir(exist_ok=True)

df.to_csv("../data/cleaned_online_retail.csv", index=False)

print("Cleaned dataset saved to ../data/cleaned_online_retail.csv")
print("Shape:", df.shape)

Cleaned dataset saved to ../data/cleaned_online_retail.csv
Shape: (779425, 8)
